# 🚀 ULM-1.7B QLoRA SFT (Google Colab Training)

이 노트북은 **Qwen3-1.7B** 모델을 구축된 울산 방언 데이터셋(Ulsan Core)으로 파인튜닝하는 Google Colab 전용 실행 노트북입니다.

### 📌 사전 준비
1. 상단 메뉴의 **[런타임] -> [런타임 유형 변경]**에서 하드웨어 가속기를 **T4 GPU** (또는 A100/L4)로 설정하세요.
2. 로컬에서 생성된 `ulsan_dataset.zip` 파일을 Colab 왼쪽 파일 패널에 업로드하세요.

In [ ]:
# 1. GPU 하드웨어 확인
!nvidia-smi

In [ ]:
# 2. 필수 라이브러리 및 ULM 패키지 설치
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate
!pip install -q -e .

In [ ]:
# 3. 업로드된 ulsan_dataset.zip 압축 해제
import os
import zipfile

dataset_dir = "data/private/ulsan_dataset"
os.makedirs(dataset_dir, exist_ok=True)

if os.path.exists("ulsan_dataset.zip"):
    with zipfile.ZipFile("ulsan_dataset.zip", "r") as z:
        z.extractall(dataset_dir)
    print(f"✓ 데이터셋 압축 해제 완료: {dataset_dir}")
    !ls -lh {dataset_dir}
elif os.path.exists(f"{dataset_dir}/train.jsonl"):
    print(f"✓ 데이터셋이 이미 존재합니다: {dataset_dir}")
else:
    print("⚠️ 'ulsan_dataset.zip' 파일을 Colab 왼쪽 파일 영역에 업로드해주세요!")

In [ ]:
# 4. ULM-1.7B QLoRA SFT 학습 실행 (백그라운드 체크포인트 저장 지원)
!python scripts/train_sft.py --config configs/sft/qwen3_1.7b_qlora.yaml

In [ ]:
# 5. 학습 완료된 어댑터로 울산 방언 추론 테스트
!python scripts/infer.py \
    --base-model Qwen/Qwen3-1.7B \
    --adapter-path outputs/qwen3-1.7b-sft \
    --prompt "오늘 날씨 참 좋다, 저녁에 밥 뭐 먹으러 갈래?" \
    --dialect-strength 2